# 📄 Phase 2: Bank Statement Row Extraction

This notebook guides you through Phase 2 of the model upgrade:
- Extract text rows from bank statement PDFs
- Label rows with entities
- Generate synthetic variations
- Prepare training data with [BANK_STATEMENT] prefix

## Goal
**Model parses bank statement rows with high accuracy**

## Step 1: Setup & Check PDF Files

Place your bank statements in: `data/raw/pdfs/statements/`

In [ ]:
from pathlib import Path
import json

# Setup directories
PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "data/raw/pdfs/statements"
LABELING_DIR = PROJECT_ROOT / "data/labeling"
TRAINING_DIR = PROJECT_ROOT / "data/training"

# Create directories
PDF_DIR.mkdir(parents=True, exist_ok=True)
LABELING_DIR.mkdir(parents=True, exist_ok=True)

# Check for PDFs
pdfs = list(PDF_DIR.glob("*.pdf")) + list(PDF_DIR.glob("*.PDF"))
print(f"📂 Found {len(pdfs)} PDF files in {PDF_DIR}")
for pdf in pdfs:
    print(f"   • {pdf.name}")

## Step 2: Extract Rows from PDFs

In [ ]:
from src.data.statement_extractor import StatementRowExtractor

extractor = StatementRowExtractor(debug=False)

all_rows = []

for pdf in pdfs:
    print(f"\n📄 Processing: {pdf.name}")
    try:
        rows, stats = extractor.extract_rows(pdf)
        all_rows.extend(rows)
        print(f"   ✅ Extracted {stats.valid_rows} rows ({stats.bank.upper()})")
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n📊 Total rows extracted: {len(all_rows)}")

## Step 3: Preview Extracted Rows

In [ ]:
import pandas as pd

# Convert to DataFrame for easy viewing
df = pd.DataFrame([r.to_dict() for r in all_rows])

# Display columns
display_cols = ['date', 'description', 'debit', 'credit', 'balance', 'bank']
cols = [c for c in display_cols if c in df.columns]

print(f"📋 Sample rows (first 10):")
df[cols].head(10)

## Step 4: Export for Manual Labeling

Export rows to JSON for manual entity labeling.

In [ ]:
output_file = LABELING_DIR / "statement_rows_unlabeled.json"
extractor.export_for_labeling(all_rows, output_file)

print(f"✅ Exported {len(all_rows)} rows to:")
print(f"   {output_file}")
print(f"\n📝 Next: Open the JSON file and add 'entities' to each row")

## Step 5: Manual Labeling Guide

For each row, add the `entities` field with:

```json
{
  "raw_text": "01-12-2025 | UPI-SWIGGY@ybl | 250.00 | | 45,230.50",
  "labeled": true,
  "entities": {
    "date": "01-12-2025",
    "description": "UPI-SWIGGY@ybl",
    "amount": "250.00",
    "type": "debit",
    "balance": "45,230.50",
    "merchant": "swiggy",
    "category": "food"
  }
}
```

**Target: Label 500+ rows**

## Step 6: Load Labeled Data

In [ ]:
# After manual labeling, load the data
labeled_file = LABELING_DIR / "statement_rows_labeled.json"

if labeled_file.exists():
    labeled_rows = extractor.load_labeled_data(labeled_file)
    labeled_count = sum(1 for r in labeled_rows if r.labeled)
    print(f"📊 Loaded {len(labeled_rows)} rows ({labeled_count} labeled)")
else:
    print(f"⚠️  Labeled file not found: {labeled_file}")
    print(f"   Rename your labeled file to: statement_rows_labeled.json")

## Step 7: Generate Synthetic Variations

In [ ]:
from src.data.statement_extractor import StatementSyntheticGenerator

generator = StatementSyntheticGenerator(seed=42)

# Generate variations from labeled data
if 'labeled_rows' in dir() and labeled_rows:
    # Filter to labeled only
    base_rows = [r for r in labeled_rows if r.labeled]
    
    # Generate 5x variations
    synthetic_rows = generator.generate_variations(
        base_rows, 
        variations_per_row=5,
        total_limit=2000
    )
    
    print(f"✅ Generated {len(synthetic_rows)} synthetic variations")
else:
    print("⚠️  Load labeled data first (Step 6)")

## Step 8: Export Training Data

In [ ]:
from src.data.statement_extractor import export_training_data

if 'synthetic_rows' in dir() and synthetic_rows:
    # Combine labeled + synthetic
    all_training = base_rows + synthetic_rows
    
    # Export
    train_file, valid_file = export_training_data(
        all_training,
        TRAINING_DIR / "statement"
    )
    
    print(f"✅ Training files created:")
    print(f"   Train: {train_file}")
    print(f"   Valid: {valid_file}")
else:
    print("⚠️  Generate synthetic data first (Step 7)")

## Step 9: Combine with Phase 1 Data

In [ ]:
# Combine Phase 1 and Phase 2 training data
phase1_train = TRAINING_DIR / "train.jsonl"
phase2_train = TRAINING_DIR / "statement_train.jsonl"
combined_train = TRAINING_DIR / "combined_train.jsonl"

if phase1_train.exists() and phase2_train.exists():
    # Read both files
    with open(phase1_train) as f:
        p1_data = f.readlines()
    with open(phase2_train) as f:
        p2_data = f.readlines()
    
    # Combine and shuffle
    import random
    combined = p1_data + p2_data
    random.shuffle(combined)
    
    with open(combined_train, 'w') as f:
        f.writelines(combined)
    
    print(f"✅ Combined training data:")
    print(f"   Phase 1: {len(p1_data)} samples")
    print(f"   Phase 2: {len(p2_data)} samples")
    print(f"   Total:   {len(combined)} samples")
else:
    print(f"⚠️  Missing files:")
    print(f"   Phase 1: {phase1_train.exists()}")
    print(f"   Phase 2: {phase2_train.exists()}")

## Step 10: Retrain Model

Run in terminal (not notebook):

```bash
cd ~/llm-mail-trainer
source venv/bin/activate

mlx_lm.lora \
    --model models/base/phi3-mini \
    --data data/training \
    --train \
    --batch-size 1 \
    --lora-layers 8 \
    --iters 800 \
    --adapter-path models/adapters/finance-lora-v4
```

Note: Use `combined_train.jsonl` and corresponding valid file.

## Step 11: Evaluate on Statement Rows

In [ ]:
from src.inference.predict import Predictor

# Load new model
predictor = Predictor(
    model_path="models/base/phi3-mini",
    adapter_path="models/adapters/finance-lora-v4"
)

# Test on statement row
test_row = "01-12-2025 | UPI-SWIGGY@ybl | 250.00 | | 45,230.50"
prompt = f"[BANK_STATEMENT] Extract financial entities from this bank statement row:\n\n{test_row}"

result = predictor.predict(email_text=prompt)
print(f"📋 Input: {test_row}")
print(f"\n🎯 Extracted:")
print(result.to_json())

## ✅ Phase 2 Checklist

- [ ] Collect bank statements (3-6 months)
- [ ] Extract text rows using pdfplumber
- [ ] Manually label 500+ rows
- [ ] Generate synthetic variations
- [ ] Add [BANK_STATEMENT] prefix to training
- [ ] Retrain model
- [ ] Test accuracy

**Deliverable: Model parses bank statement rows**